# 🇬🇧 English ↔ Hindi Seq2Seq Translator 🇮🇳

A bidirectional neural machine translation model using LSTM Encoder-Decoder architecture.

## Architecture at a Glance

```
 ENCODER                              DECODER
┌───┐ ┌───┐ ┌───┐ ┌───┐            ┌───┐ ┌───┐ ┌───┐ ┌───┐
│ I │→│ am│→│ a │→│boy│──(h,c)──→│मैं│→│एक │→│लड़का│→│हूँ│
└───┘ └───┘ └───┘ └───┘            └───┘ └───┘ └───┘ └───┘
  LSTM cells (encoding)    context   LSTM cells (decoding)
```

| Concept | What it means |
|---------|--------------|
| **Encoder** | Reads the English sentence token by token → produces a context vector `(h, c)` |
| **Decoder** | Takes the context vector → generates Hindi tokens one by one |
| **Teacher Forcing** | During training, the decoder sees the *correct* previous token (not its own guess) |
| **Start / End tokens** | `\t` = start of Hindi output, `\n` = stop signal |

**Stack:** TensorFlow / Keras · NumPy · Pandas · NLTK · Flask (backend) · React (frontend)


---
## Section 1 — Setup: Install & Import

In [ ]:
# Install dependencies (run once in Colab / Kaggle)
!pip install tensorflow numpy pandas nltk openpyxl xlrd --quiet

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import re, os, json, pickle, warnings

from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Dense, Embedding,
    AdditiveAttention, Concatenate
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

import nltk
from nltk.tokenize import word_tokenize
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Download NLTK tokenizer data
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

print(f'TensorFlow : {tf.__version__}')
print(f'NumPy      : {np.__version__}')
print(f'Pandas     : {pd.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')
print('✅ Libraries ready!')

In [ ]:
import psutil

ram_gb = psutil.virtual_memory().total / (1024**3)

if ram_gb < 8:
    MAX_SAMPLES = 30_000
    print(f'⚠️  {ram_gb:.1f} GB RAM detected — capping at 30k samples')
elif ram_gb < 16:
    MAX_SAMPLES = 65_000
    print(f'✅ {ram_gb:.1f} GB RAM — using up to 65k samples')
else:
    MAX_SAMPLES = 65_000
    print(f'✅ {ram_gb:.1f} GB RAM — comfortable, using up to 65k samples')

---
## Section 2 — Load & Explore the Dataset

In [ ]:
DATASET_PATH = 'hindi_english_parallel.xls'

# Try both Excel engines (.xls vs .xlsx)
try:
    df = pd.read_excel(DATASET_PATH, engine='xlrd')
except Exception:
    df = pd.read_excel(DATASET_PATH, engine='openpyxl')

print(f'Shape   : {df.shape}')
print(f'Columns : {list(df.columns)}')
df.head(5)

In [ ]:
# Auto-detect English and Hindi columns by name
cols = df.columns.tolist()
eng_col = hin_col = None

for c in cols:
    cl = str(c).lower().strip()
    if 'eng' in cl or cl == 'en' or 'source' in cl:
        eng_col = c
    elif 'hin' in cl or cl == 'hi' or 'target' in cl:
        hin_col = c

# Fallback: assume first = English, second = Hindi
eng_col = eng_col or cols[0]
hin_col = hin_col or cols[1]

print(f'English column : "{eng_col}"')
print(f'Hindi column   : "{hin_col}"')

df = df[[eng_col, hin_col]].copy()
df.columns = ['english', 'hindi']
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'\nRows after dropping nulls: {len(df)}')
df.head(5)

In [ ]:
# Quick EDA — sentence length statistics
df['eng_len'] = df['english'].apply(lambda x: len(str(x).split()))
df['hin_len'] = df['hindi'].apply(lambda x:   len(str(x).split()))

print('English sentence lengths:')
print(df['eng_len'].describe().to_string())
print('\nHindi sentence lengths:')
print(df['hin_len'].describe().to_string())

print('\n── Sample Pairs ──')
for i in range(min(3, len(df))):
    print(f'  EN: {df.iloc[i]["english"]}')
    print(f'  HI: {df.iloc[i]["hindi"]}')
    print()

---
## Section 3 — Text Preprocessing

We need to:
1. **Clean** both languages (remove noise, normalise whitespace)
2. **Add start/end tokens** to Hindi sequences so the decoder knows when to start and stop
3. **Filter** very long sentences (they waste memory and slow training)
4. **Cap** the dataset at `MAX_SAMPLES` (set in Section 1 based on your RAM)


In [ ]:
def clean_english(text):
    """Lowercase, remove punctuation, tokenize with NLTK."""
    text = str(text).lower().strip()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)   # strip punctuation
    text = re.sub(r'\s+', ' ', text).strip()       # collapse whitespace
    return ' '.join(word_tokenize(text))


def clean_hindi(text):
    """Keep only Devanagari Unicode block, digits, and spaces."""
    text = str(text).strip()
    text = re.sub(r'[^\u0900-\u097F0-9\s]', '', text)
    return re.sub(r'\s+', ' ', text).strip()


# Apply cleaning
df['english_clean'] = df['english'].apply(clean_english)
df['hindi_clean']   = df['hindi'].apply(clean_hindi)

# Add START (\t) and END (\n) tokens to Hindi
#   hindi_input  = decoder INPUT  → starts with \t (teacher-forcing step)
#   hindi_target = decoder TARGET → ends   with \n (what the model learns to produce)
df['hindi_input']  = df['hindi_clean'].apply(lambda x: '\t ' + x)
df['hindi_target'] = df['hindi_clean'].apply(lambda x: x + '\n')

print('Cleaning done ✅')
for i in range(min(3, len(df))):
    print(f'  EN clean    : {df.iloc[i]["english_clean"]}')
    print(f'  HI input    : {df.iloc[i]["hindi_input"]}')
    print(f'  HI target   : {df.iloc[i]["hindi_target"]}')
    print()

In [ ]:
# Filter sentences that are too long (saves RAM and speeds up training)
MAX_ENG_LEN = 30
MAX_HIN_LEN = 30

df = df[
    (df['english_clean'].apply(lambda x: len(x.split())) <= MAX_ENG_LEN) &
    (df['hindi_clean'].apply(lambda x:   len(x.split())) <= MAX_HIN_LEN)
].copy().reset_index(drop=True)

print(f'After length filter : {len(df)} rows')

# Respect the RAM-aware MAX_SAMPLES set in Section 1
MAX_SAMPLES = min(globals().get('MAX_SAMPLES', 50_000), len(df))
df = df.head(MAX_SAMPLES).copy()
print(f'Training with       : {len(df)} sentence pairs')

---
## Section 4 — Tokenization & Sequence Preparation

**Tokenization** converts words → integer IDs.
**Padding** makes all sequences the same length (required by the model).

```
"I am a boy"  →  [4, 12, 2, 87]  →  [4, 12, 2, 87, 0, 0, 0]  (padded to max_len)
```


In [ ]:
# ── English tokenizer (encoder input) ──────────────────────────────
eng_tokenizer = Tokenizer(filters='', oov_token='<OOV>')
eng_tokenizer.fit_on_texts(df['english_clean'])

eng_sequences   = eng_tokenizer.texts_to_sequences(df['english_clean'])
eng_vocab_size  = len(eng_tokenizer.word_index) + 1
max_eng_seq_len = max(len(s) for s in eng_sequences)

encoder_input_data = pad_sequences(eng_sequences, maxlen=max_eng_seq_len, padding='post')

print(f'English vocab size    : {eng_vocab_size}')
print(f'Max English seq len   : {max_eng_seq_len}')
print(f'Encoder input shape   : {encoder_input_data.shape}')

# ── Hindi tokenizer (decoder input & target) ────────────────────────
hin_tokenizer = Tokenizer(filters='', oov_token='<OOV>')
# Fit on both input and target so \t and \n tokens are captured
hin_tokenizer.fit_on_texts(df['hindi_input'].tolist() + df['hindi_target'].tolist())

hin_input_seqs  = hin_tokenizer.texts_to_sequences(df['hindi_input'])
hin_target_seqs = hin_tokenizer.texts_to_sequences(df['hindi_target'])

hin_vocab_size  = len(hin_tokenizer.word_index) + 1
max_hin_seq_len = max(
    max(len(s) for s in hin_input_seqs),
    max(len(s) for s in hin_target_seqs)
)

decoder_input_data  = pad_sequences(hin_input_seqs,  maxlen=max_hin_seq_len, padding='post')
decoder_target_data = np.array(
    pad_sequences(hin_target_seqs, maxlen=max_hin_seq_len, padding='post')
)

print(f'\nHindi vocab size      : {hin_vocab_size}')
print(f'Max Hindi seq len     : {max_hin_seq_len}')
print(f'Decoder input shape   : {decoder_input_data.shape}')
print(f'Decoder target shape  : {decoder_target_data.shape}')
print(f'Target memory usage   : {decoder_target_data.nbytes / 1e6:.1f} MB  (no one-hot = RAM safe ✅)')

# Reverse lookup: index → word (needed for decoding predictions back to text)
eng_index_to_word = {idx: word for word, idx in eng_tokenizer.word_index.items()}
hin_index_to_word = {idx: word for word, idx in hin_tokenizer.word_index.items()}

---
## Section 5 — Build the EN→HI Seq2Seq Model

```
┌──────────────────────────────────────────────────────────────┐
│                      TRAINING MODEL                          │
│                                                              │
│  Encoder:                                                    │
│  [English tokens] → Embedding → LSTM → (state_h, state_c)  │
│                                              │        │      │
│  Decoder:                                    ▼        ▼      │
│  [Hindi tokens]   → Embedding → LSTM(initial_state)         │
│                                     → Dense(softmax)         │
│                                     → Hindi word probs       │
└──────────────────────────────────────────────────────────────┘
```


In [ ]:
# ── Hyperparameters ─────────────────────────────────────────────────
EMBEDDING_DIM    = 256   # width of word embedding vectors
LATENT_DIM       = 512   # LSTM hidden units — the "memory" size
BATCH_SIZE       = 128   # larger = faster epochs, same quality on GPU
EPOCHS           = 50    # EarlyStopping will cut this short
VALIDATION_SPLIT = 0.1   # 10% of data held out for validation

print(f'Embedding dim    : {EMBEDDING_DIM}')
print(f'Latent dim       : {LATENT_DIM}')
print(f'Batch size       : {BATCH_SIZE}')
print(f'Max epochs       : {EPOCHS}')

In [ ]:
# ── Encoder ──────────────────────────────────────────────────────────
encoder_inputs = Input(shape=(max_eng_seq_len,), name='encoder_input')

enc_embedding = Embedding(
    input_dim=eng_vocab_size,
    output_dim=EMBEDDING_DIM,
    mask_zero=True,
    name='encoder_embedding'
)(encoder_inputs)

encoder_lstm = LSTM(
    LATENT_DIM,
    return_sequences=True,   # needed so inference encoder can return all outputs
    return_state=True,
    name='encoder_lstm',
    use_cudnn=False
)

# encoder_outputs = all hidden states (used by attention at inference)
# state_h, state_c = final states = the context vector passed to decoder
encoder_outputs, state_h, state_c = encoder_lstm(enc_embedding)
encoder_states = [state_h, state_c]

print('✅ Encoder built')
print(f'   encoder_outputs : {encoder_outputs.shape}')
print(f'   state_h         : {state_h.shape}')
print(f'   state_c         : {state_c.shape}')

In [ ]:
# ── Decoder ──────────────────────────────────────────────────────────
decoder_inputs = Input(shape=(max_hin_seq_len,), name='decoder_input')

dec_embedding_layer = Embedding(
    input_dim=hin_vocab_size,
    output_dim=EMBEDDING_DIM,
    mask_zero=True,
    name='decoder_embedding'
)
dec_embedding = dec_embedding_layer(decoder_inputs)

decoder_lstm = LSTM(
    LATENT_DIM,
    return_sequences=True,
    return_state=True,
    name='decoder_lstm',
    use_cudnn=False
)

# The KEY connection: decoder starts from encoder's final states
decoder_outputs, _, _ = decoder_lstm(dec_embedding, initial_state=encoder_states)

# Dense layer: map each decoder output → probability over Hindi vocabulary
decoder_dense = Dense(hin_vocab_size, activation='softmax', name='decoder_output')
decoder_outputs = decoder_dense(decoder_outputs)

print('✅ Decoder built')
print(f'   Output shape : {decoder_outputs.shape}')

In [ ]:
# ── Compile the training model ───────────────────────────────────────
model = Model(
    inputs=[encoder_inputs, decoder_inputs],
    outputs=decoder_outputs,
    name='Seq2Seq_EN_to_HI'
)

# sparse_categorical_crossentropy: targets are integers (not one-hot) → saves RAM
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Optional: visualise the model graph (requires graphviz)
try:
    keras.utils.plot_model(model, to_file='seq2seq_model.png', show_shapes=True, dpi=100)
    from IPython.display import Image
    display(Image('seq2seq_model.png'))
except Exception as e:
    print(f'Graph plot skipped (graphviz not installed): {e}')

---
## Section 6 — Train the EN→HI Model

**EarlyStopping** watches `val_loss` and stops training once it stops improving (patience = 5 epochs).
**ModelCheckpoint** saves the best weights automatically so you never lose a good run.


In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        'best_seq2seq_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

print('🚀 Starting EN→HI training...\n')

history = model.fit(
    x=[encoder_input_data, decoder_input_data],
    y=decoder_target_data,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=VALIDATION_SPLIT,
    callbacks=callbacks,
    verbose=1
)

print('\n✅ Training complete!')

In [ ]:
# Plot loss and accuracy curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['loss'],     label='Train Loss',     linewidth=2)
ax1.plot(history.history['val_loss'], label='Val Loss',       linewidth=2)
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history.history['accuracy'],     label='Train Accuracy', linewidth=2)
ax2.plot(history.history['val_accuracy'], label='Val Accuracy',   linewidth=2)
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()
print('📊 Saved training_history.png')

---
## Section 7 — Build Inference Models

During training we used **Teacher Forcing** (decoder saw the correct previous token).
At inference time we must **loop** — each predicted token becomes the input for the next step.

We need two separate models for this:

| Model | Input | Output |
|-------|-------|--------|
| `encoder_model` | English sentence | `(encoder_outputs, state_h, state_c)` |
| `decoder_model` | One Hindi token + previous states | Next token probabilities + new states |

The decoder loop runs until `\n` (end token) is predicted or `max_hin_seq_len` is reached.


In [ ]:
# ── Encoder inference model ──────────────────────────────────────────
# Returns all encoder outputs + final states so app.py beam_search can unpack:
#   enc_outs, h, c = encoder_model.predict(...)
encoder_model = Model(
    inputs=encoder_inputs,
    outputs=[encoder_outputs, state_h, state_c],
    name='encoder_inference'
)

# ── Decoder inference model ───────────────────────────────────────────
# Takes one token at a time + previous LSTM states → next token + new states.
# The encoder_outputs input is included to keep the same 4-input signature
# as the reverse (HI→EN) model, so app.py beam_search() works for both directions.
decoder_state_input_h  = Input(shape=(LATENT_DIM,),                   name='decoder_state_h')
decoder_state_input_c  = Input(shape=(LATENT_DIM,),                   name='decoder_state_c')
decoder_enc_out_input  = Input(shape=(max_eng_seq_len, LATENT_DIM),   name='decoder_enc_out')
decoder_single_input   = Input(shape=(1,),                            name='decoder_single_token')

dec_emb_single = dec_embedding_layer(decoder_single_input)

decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    dec_emb_single,
    initial_state=[decoder_state_input_h, decoder_state_input_c]
)
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

# Pass encoder_outputs through a Lambda so Keras includes it in the graph
# (it is not used in computation here — only present for API symmetry with HI→EN)
enc_out_passthrough = tf.keras.layers.Lambda(
    lambda x: x, name='enc_out_passthrough'
)(decoder_enc_out_input)

decoder_model = Model(
    inputs=[decoder_single_input, decoder_enc_out_input,
            decoder_state_input_h, decoder_state_input_c],
    outputs=[decoder_outputs_inf, state_h_inf, state_c_inf, enc_out_passthrough],
    name='decoder_inference'
)

print('✅ Encoder inference model built')
print('✅ Decoder inference model built')

# Save immediately — app.py looks for these exact filenames
encoder_model.save('att_encoder_inference.keras')
decoder_model.save('att_decoder_inference.keras')
print('✅ Saved: att_encoder_inference.keras, att_decoder_inference.keras')

---
## Section 8 — Translation Function (EN→HI)

**How it works step by step:**
1. Clean and tokenize the English input
2. Run it through `encoder_model` → get context vector `(enc_outs, h, c)`
3. Feed `\t` (start token) into `decoder_model`
4. Take the predicted word, feed it back in as the next input
5. Repeat until `\n` (end token) is predicted or max length is reached


In [ ]:
def translate_sentence(input_sentence):
    """Translate an English sentence to Hindi (greedy decoding)."""

    # Step 1 — preprocess
    cleaned = clean_english(input_sentence)
    padded  = pad_sequences(
        eng_tokenizer.texts_to_sequences([cleaned]),
        maxlen=max_eng_seq_len, padding='post'
    )

    # Step 2 — encode: returns [encoder_outputs, state_h, state_c]
    states = list(encoder_model.predict(padded, verbose=0))

    # Step 3 — initialise decoder with start token
    start_idx = hin_tokenizer.word_index.get('\t')
    end_idx   = hin_tokenizer.word_index.get('\n')

    if start_idx is None:
        print('⚠️  Start token (\\t) not found in Hindi vocabulary!')
        return ''

    token = np.zeros((1, 1))
    token[0, 0] = start_idx

    # Step 4 — decode loop
    words = []
    for _ in range(max_hin_seq_len):
        # decoder_model outputs: [probs, new_h, new_c, enc_out_passthrough]
        probs, new_h, new_c, _ = decoder_model.predict([token] + states, verbose=0)

        predicted_idx = int(np.argmax(probs[0, -1, :]))

        # Stop on end token or padding token
        if predicted_idx in (0, end_idx):
            break

        word = hin_index_to_word.get(predicted_idx, '')
        if word and word not in ('\t', '\n'):
            words.append(word)

        # Feed prediction back and update states
        token[0, 0] = predicted_idx
        states[1] = new_h   # update state_h
        states[2] = new_c   # update state_c

    return ' '.join(words)


print('✅ translate_sentence() ready')

---
## Section 9 — Test EN→HI Translations

In [ ]:
# Sanity check on training data
print('=' * 65)
print('  TRAINING DATA CHECK (15 random samples)')
print('=' * 65)

for idx in np.random.choice(len(df), min(15, len(df)), replace=False):
    eng  = df.iloc[idx]['english']
    real = df.iloc[idx]['hindi']
    pred = translate_sentence(eng)
    print(f'\n  🇬🇧 {eng}')
    print(f'  🇮🇳 Expected : {real}')
    print(f'  🤖 Predicted: {pred}')
    print(f'  {"─"*58}')

In [ ]:
# Test on custom sentences
custom = [
    'How are you',
    'My name is Raj',
    'I am going to school',
    'What is your name',
    'India is a great country',
    'I love my mother',
    'The weather is good today',
    'Please give me water',
    'Thank you very much',
    'Good morning',
]

print('=' * 65)
print('  CUSTOM SENTENCE TRANSLATIONS')
print('=' * 65)
for s in custom:
    print(f'  🇬🇧 {s}')
    print(f'  🇮🇳 {translate_sentence(s)}')
    print()

In [ ]:
# ✏️ Change this sentence and re-run the cell to try any input
user_input = 'I am happy to learn Hindi'
print(f'EN: {user_input}')
print(f'HI: {translate_sentence(user_input)}')

---
## Section 10 — Save EN→HI Model & Tokenizers

In [ ]:
model.save('seq2seq_eng_to_hindi.keras')

with open('eng_tokenizer.pkl', 'wb') as f: pickle.dump(eng_tokenizer, f)
with open('hin_tokenizer.pkl', 'wb') as f: pickle.dump(hin_tokenizer, f)

# Save config (app.py reads this to know sequence lengths)
en_hi_config = {
    'EMBEDDING_DIM'   : EMBEDDING_DIM,
    'LATENT_DIM'      : LATENT_DIM,
    'max_eng_seq_len' : max_eng_seq_len,
    'max_hin_seq_len' : max_hin_seq_len,
    'eng_vocab_size'  : eng_vocab_size,
    'hin_vocab_size'  : hin_vocab_size,
}
with open('model_config.json', 'w') as f:
    json.dump(en_hi_config, f, indent=2)

print('✅ EN→HI assets saved:')
print('   ├── seq2seq_eng_to_hindi.keras')
print('   ├── att_encoder_inference.keras')
print('   ├── att_decoder_inference.keras')
print('   ├── eng_tokenizer.pkl')
print('   ├── hin_tokenizer.pkl')
print('   └── model_config.json')

---
## Section 11 — Hindi → English (Reverse Model): Preprocessing

We reuse the same cleaned dataset but **swap encoder and decoder roles**:

| Direction | Encoder input | Decoder output |
|-----------|--------------|----------------|
| EN → HI (above) | English | Hindi |
| HI → EN (this section) | Hindi | English |


In [ ]:
# For HI→EN, English now plays the decoder role → needs start/end tokens
df['english_input']  = df['english_clean'].apply(lambda x: '\t ' + x)
df['english_target'] = df['english_clean'].apply(lambda x: x + ' \n')

# ── Hindi as ENCODER input ───────────────────────────────────────────
rev_hin_tokenizer = Tokenizer(filters='', oov_token='<OOV>')
rev_hin_tokenizer.fit_on_texts(df['hindi_clean'])

rev_hin_seqs        = rev_hin_tokenizer.texts_to_sequences(df['hindi_clean'])
rev_hin_vocab_size  = len(rev_hin_tokenizer.word_index) + 1
max_rev_hin_seq_len = max(len(s) for s in rev_hin_seqs)

rev_encoder_input_data = pad_sequences(rev_hin_seqs, maxlen=max_rev_hin_seq_len, padding='post')

print(f'Hindi (encoder) vocab : {rev_hin_vocab_size}')
print(f'Max Hindi seq len     : {max_rev_hin_seq_len}')
print(f'Rev encoder shape     : {rev_encoder_input_data.shape}')

# ── English as DECODER input/target ─────────────────────────────────
rev_eng_tokenizer = Tokenizer(filters='', oov_token='<OOV>')
rev_eng_tokenizer.fit_on_texts(df['english_input'].tolist() + df['english_target'].tolist())

rev_eng_input_seqs  = rev_eng_tokenizer.texts_to_sequences(df['english_input'])
rev_eng_target_seqs = rev_eng_tokenizer.texts_to_sequences(df['english_target'])

rev_eng_vocab_size  = len(rev_eng_tokenizer.word_index) + 1
max_rev_eng_seq_len = max(
    max(len(s) for s in rev_eng_input_seqs),
    max(len(s) for s in rev_eng_target_seqs)
)

rev_decoder_input_data  = pad_sequences(rev_eng_input_seqs,  maxlen=max_rev_eng_seq_len, padding='post')
rev_decoder_target_data = np.array(
    pad_sequences(rev_eng_target_seqs, maxlen=max_rev_eng_seq_len, padding='post')
)

rev_eng_index_to_word = {idx: word for word, idx in rev_eng_tokenizer.word_index.items()}

print(f'\nEnglish (decoder) vocab : {rev_eng_vocab_size}')
print(f'Max English seq len     : {max_rev_eng_seq_len}')
print(f'Rev decoder input shape : {rev_decoder_input_data.shape}')
print('✅ Reverse preprocessing complete!')

---
## Section 12 — Build the HI→EN Model (with Attention)

This model adds **Bahdanau (Additive) Attention** on top of the basic Seq2Seq.
Instead of the decoder relying only on the encoder's final state, it can
look at *all* encoder outputs and learn which input words matter most
at each decoding step.


In [ ]:
EMBEDDING_DIM_REV = 256
LATENT_DIM_REV    = 512

# ── Reverse Encoder (Hindi input) ────────────────────────────────────
rev_enc_inputs = Input(shape=(max_rev_hin_seq_len,), name='rev_enc_input')

rev_enc_emb = Embedding(
    rev_hin_vocab_size, EMBEDDING_DIM_REV,
    mask_zero=True, name='rev_enc_emb'
)(rev_enc_inputs)

rev_enc_lstm = LSTM(
    LATENT_DIM_REV,
    return_sequences=True,   # attention needs all timestep outputs
    return_state=True,
    name='rev_enc_lstm',
    use_cudnn=False
)
rev_enc_out, rev_enc_h, rev_enc_c = rev_enc_lstm(rev_enc_emb)

# ── Reverse Decoder (English output) with Attention ──────────────────
rev_dec_inputs    = Input(shape=(max_rev_eng_seq_len,), name='rev_dec_input')
rev_dec_emb_layer = Embedding(rev_eng_vocab_size, EMBEDDING_DIM_REV, mask_zero=True, name='rev_dec_emb')
rev_dec_emb       = rev_dec_emb_layer(rev_dec_inputs)

rev_dec_lstm = LSTM(LATENT_DIM_REV, return_sequences=True, return_state=True,
                    name='rev_dec_lstm', use_cudnn=False)
rev_dec_out, _, _ = rev_dec_lstm(rev_dec_emb, initial_state=[rev_enc_h, rev_enc_c])

# Attention: decoder output queries encoder outputs
rev_attention = AdditiveAttention(name='rev_attention')
rev_context   = rev_attention([rev_dec_out, rev_enc_out])  # [query, value]

# Concatenate decoder output with its attention-weighted context
rev_concat = Concatenate(name='rev_concat')([rev_dec_out, rev_context])

rev_dense  = Dense(rev_eng_vocab_size, activation='softmax', name='rev_output')
rev_output = rev_dense(rev_concat)

# ── Compile ───────────────────────────────────────────────────────────
rev_model = Model(
    inputs=[rev_enc_inputs, rev_dec_inputs],
    outputs=rev_output,
    name='Seq2Seq_HI_to_EN'
)
rev_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
rev_model.summary()
print('\n✅ HI→EN model built!')

---
## Section 13 — Train the HI→EN Model

In [ ]:
rev_callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_rev_model.keras', monitor='val_loss', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
]

print('🚀 Training HI→EN model...\n')

rev_history = rev_model.fit(
    x=[rev_encoder_input_data, rev_decoder_input_data],
    y=rev_decoder_target_data,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=VALIDATION_SPLIT,
    callbacks=rev_callbacks,
    verbose=1
)

print('\n✅ HI→EN training complete!')

---
## Section 14 — Build HI→EN Inference Models

In [ ]:
# ── Reverse encoder inference ────────────────────────────────────────
rev_enc_model = Model(
    inputs=rev_enc_inputs,
    outputs=[rev_enc_out, rev_enc_h, rev_enc_c],
    name='rev_encoder_inference'
)

# ── Reverse decoder inference ─────────────────────────────────────────
rev_dec_single  = Input(shape=(1,),                                  name='rev_dec_token')
rev_dec_enc_out = Input(shape=(max_rev_hin_seq_len, LATENT_DIM_REV), name='rev_dec_enc_out')
rev_dec_h_in    = Input(shape=(LATENT_DIM_REV,),                     name='rev_dec_h')
rev_dec_c_in    = Input(shape=(LATENT_DIM_REV,),                     name='rev_dec_c')

rev_single_emb              = rev_dec_emb_layer(rev_dec_single)
rev_dec_out_inf, rev_h_inf, rev_c_inf = rev_dec_lstm(
    rev_single_emb, initial_state=[rev_dec_h_in, rev_dec_c_in]
)
rev_ctx_inf = rev_attention([rev_dec_out_inf, rev_dec_enc_out])
# Fresh Concatenate layer — do NOT reuse rev_concat (was built with training shapes)
rev_cat_inf = Concatenate(name='rev_concat_inf')([rev_dec_out_inf, rev_ctx_inf])
rev_out_inf = rev_dense(rev_cat_inf)

rev_dec_model = Model(
    inputs=[rev_dec_single, rev_dec_enc_out, rev_dec_h_in, rev_dec_c_in],
    outputs=[rev_out_inf, rev_h_inf, rev_c_inf],
    name='rev_decoder_inference'
)

print('✅ Reverse encoder inference model built')
print('✅ Reverse decoder inference model built')

---
## Section 15 — HI→EN Translation Function (Beam Search)

**Beam Search** keeps the top-`k` candidate translations at each step
instead of always picking just the single best word (greedy).
This produces more natural, globally-coherent translations.


In [ ]:
REV_START = rev_eng_tokenizer.word_index.get('\t', 1)
REV_END   = rev_eng_tokenizer.word_index.get('\n', 2)


def translate_hindi_to_english(hindi_sentence, beam_width=5):
    """Translate a Hindi sentence to English using beam search + attention."""

    # Preprocess
    cleaned = clean_hindi(hindi_sentence)
    padded  = pad_sequences(
        rev_hin_tokenizer.texts_to_sequences([cleaned]),
        maxlen=max_rev_hin_seq_len, padding='post'
    )

    # Encode
    enc_outs, h, c = rev_enc_model.predict(padded, verbose=0)

    # Beam search
    # Each beam: (negative_log_prob_score, token_ids, state_h, state_c)
    beams     = [(0.0, [REV_START], h, c)]
    completed = []

    for _ in range(max_rev_eng_seq_len):
        candidates = []
        for score, ids, bh, bc in beams:
            if ids[-1] == REV_END:
                completed.append((score, ids, bh, bc))
                continue
            token = np.zeros((1, 1))
            token[0, 0] = ids[-1]
            probs, nh, nc = rev_dec_model.predict([token, enc_outs, bh, bc], verbose=0)
            top_k = np.argsort(probs[0, -1, :])[-beam_width:][::-1]
            for tok in top_k:
                lp = np.log(probs[0, -1, tok] + 1e-10)
                candidates.append((score - lp, ids + [int(tok)], nh, nc))

        if not candidates:
            break
        candidates.sort(key=lambda x: x[0])
        beams = candidates[:beam_width]
        if all(b[1][-1] == REV_END for b in beams):
            completed.extend(beams)
            break

    best  = min(completed or beams, key=lambda x: x[0])
    words = [
        rev_eng_index_to_word[idx]
        for idx in best[1][1:]
        if idx not in (0, REV_END) and rev_eng_index_to_word.get(idx, '') not in ('\t', '\n')
    ]
    return ' '.join(words)


print('✅ translate_hindi_to_english() ready')

---
## Section 16 — Test Both Directions

In [ ]:
test_pairs = [
    ('How are you',        'आप कैसे हैं'),
    ('I love my mother',   'मुझे अपनी माँ से प्यार है'),
    ('Good morning',       'सुप्रभात'),
    ('What is your name',  'आपका नाम क्या है'),
    ('India is great',     'भारत महान है'),
]

print('=' * 65)
print('  BIDIRECTIONAL TRANSLATION TEST')
print('=' * 65)

for en, hi in test_pairs:
    print(f'\n  🇬🇧 EN original : {en}')
    print(f'  🇮🇳 EN → HI     : {translate_sentence(en)}')
    print(f'  🇮🇳 HI original : {hi}')
    print(f'  🇬🇧 HI → EN     : {translate_hindi_to_english(hi)}')
    print(f'  {"─"*58}')

---
## Section 17 — Save Everything & Download

In [ ]:
# Save HI→EN models
rev_model.save('seq2seq_hin_to_eng.keras')
rev_enc_model.save('rev_encoder_inference.keras')
rev_dec_model.save('rev_decoder_inference.keras')

with open('rev_hin_tokenizer.pkl', 'wb') as f: pickle.dump(rev_hin_tokenizer, f)
with open('rev_eng_tokenizer.pkl', 'wb') as f: pickle.dump(rev_eng_tokenizer, f)

# Save combined config (app.py reads this for both directions)
config_v3 = {
    'EN_TO_HI': {
        'EMBEDDING_DIM'   : EMBEDDING_DIM,
        'LATENT_DIM'      : LATENT_DIM,
        'max_eng_seq_len' : max_eng_seq_len,
        'max_hin_seq_len' : max_hin_seq_len,
        'eng_vocab_size'  : eng_vocab_size,
        'hin_vocab_size'  : hin_vocab_size,
    },
    'HI_TO_EN': {
        'EMBEDDING_DIM'   : EMBEDDING_DIM_REV,
        'LATENT_DIM'      : LATENT_DIM_REV,
        'max_hin_seq_len' : max_rev_hin_seq_len,
        'max_eng_seq_len' : max_rev_eng_seq_len,
        'hin_vocab_size'  : rev_hin_vocab_size,
        'eng_vocab_size'  : rev_eng_vocab_size,
    }
}
with open('model_config_v3.json', 'w') as f:
    json.dump(config_v3, f, indent=2)

print('✅ All files saved!')
print()
print('EN→HI files:')
print('   ├── seq2seq_eng_to_hindi.keras')
print('   ├── att_encoder_inference.keras')
print('   ├── att_decoder_inference.keras')
print('   ├── eng_tokenizer.pkl')
print('   └── hin_tokenizer.pkl')
print()
print('HI→EN files:')
print('   ├── seq2seq_hin_to_eng.keras')
print('   ├── rev_encoder_inference.keras')
print('   ├── rev_decoder_inference.keras')
print('   ├── rev_hin_tokenizer.pkl')
print('   └── rev_eng_tokenizer.pkl')
print()
print('Shared:')
print('   └── model_config_v3.json')

In [ ]:
# # Save to Google Drive so files survive Colab session disconnect
# from google.colab import drive
# drive.mount('/content/drive')

# import shutil, os

# DRIVE_SAVE_DIR = '/content/drive/MyDrive/translator_model'
# os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

# files_to_save = [
#     'seq2seq_eng_to_hindi.keras',
#     'att_encoder_inference.keras',
#     'att_decoder_inference.keras',
#     'eng_tokenizer.pkl',
#     'hin_tokenizer.pkl',
#     'seq2seq_hin_to_eng.keras',
#     'rev_encoder_inference.keras',
#     'rev_decoder_inference.keras',
#     'rev_hin_tokenizer.pkl',
#     'rev_eng_tokenizer.pkl',
#     'model_config_v3.json',
#     'training_history.png',
# ]

# for fname in files_to_save:
#     if os.path.exists(fname):
#         shutil.copy(fname, DRIVE_SAVE_DIR)
#         print(f'  ✅ {fname}')
#     else:
#         print(f'  ⚠️  {fname} not found — skipping')

# print(f'\n📁 All saved to {DRIVE_SAVE_DIR}')

---
## Summary

| Component | EN → HI | HI → EN |
|-----------|---------|---------|
| **Architecture** | Seq2Seq LSTM | Seq2Seq LSTM + Bahdanau Attention |
| **Decoding** | Greedy | Beam Search (k=5) |
| **Encoder input** | English tokens | Hindi tokens |
| **Decoder output** | Hindi tokens | English tokens |
| **Inference models** | `att_encoder/decoder_inference.keras` | `rev_encoder/decoder_inference.keras` |
| **Tokenizers** | `eng_tokenizer.pkl`, `hin_tokenizer.pkl` | `rev_hin_tokenizer.pkl`, `rev_eng_tokenizer.pkl` |

Both directions are served by `app.py` via `POST /translate` and selectable in the React frontend via the 🔄 flip button.

### Tips to Improve Translation Quality
1. **Attention on EN→HI too** — add `AdditiveAttention` to the EN→HI decoder as well
2. **More data** — quality scales with dataset size (aim for 100k+ sentence pairs)
3. **Subword tokenization** — use SentencePiece / BPE to handle rare and unseen words
4. **Transformer** — replace LSTMs with self-attention ("Attention Is All You Need")

---
*Built with TensorFlow · Keras · NumPy · Pandas · NLTK · Flask · React*
